# Day 080 — Solution: A ReAct Reasoning Agent

In [ ]:
_SRC = '"""react_agent.py — Day 080: The Agent Loop (ReAct).\n\nDay 79 built an agent that emitted a bare JSON action each turn. ReAct\n(Reason + Act) makes the model *think out loud* first: every turn is a\nThought, then an Action with an Input, and the loop feeds back an Observation.\nReasoning before acting measurably improves multi-step tool use.\n\nThe ReAct turn format:\n    Thought: <reasoning about what to do next>\n    Action: <one tool name>\n    Input: {"<param>": "<value>"}\n...and, when done:\n    Thought: <final reasoning>\n    Final Answer: <the answer>\n\nPieces (ReAct-specific parts are new on Day 080; tools reuse Day 79):\n  safe_calculate / _lookup / DEFAULT_TOOLS / build_tool_descriptions  (Day 79)\n  safe_parse_json          - tolerant JSON parser                     (Day 79)\n  parse_react_step         - parse one Thought/Action/Input step (never raises)\n  format_step              - render an action step back into ReAct text\n  format_observation       - render a tool result as an Observation line\n  build_react_prompt       - assemble the messages, including the scratchpad\n  execute_action           - run the tool named in a step\n  call_llm                 - Ollama wrapper with llm_fn injection       (Day 79)\n  run_react_agent          - the ReAct loop (bounded by max_iterations)\n  ReactAgent               - a stateful ReAct agent\n\nSetup:\n    pip install ollama\n    ollama pull llama3.2\n"""\nimport ast\nimport json\nimport operator\n\n# ── tools reused from Day 79: a safe calculator + a fact-lookup tool ──────────\n_OPS = {\n    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,\n    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,\n    ast.USub: operator.neg, ast.UAdd: operator.pos,\n}\n\n\ndef _eval_node(node):\n    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):\n        return node.value\n    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:\n        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))\n    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:\n        return _OPS[type(node.op)](_eval_node(node.operand))\n    raise ValueError("unsupported expression")\n\n\ndef safe_calculate(expression):\n    """Evaluate arithmetic without eval() (see Day 79)."""\n    return _eval_node(ast.parse(expression, mode="eval").body)\n\n\n_FACTS = {\n    "speed of light": "299792458 m/s",\n    "pi": "3.14159",\n    "earth radius": "6371 km",\n    "days in a year": "365",\n}\n\n\ndef _lookup(args):\n    query = str(args.get("query", "")).lower().strip()\n    for key, value in _FACTS.items():\n        if query and (query in key or key in query):\n            return value\n    return "No result found for " + repr(args.get("query", ""))\n\n\nDEFAULT_TOOLS = {\n    "calculator": {\n        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",\n        "parameters": {"expression": "string - the arithmetic to evaluate"},\n        "fn": lambda args: str(safe_calculate(args["expression"])),\n    },\n    "lookup": {\n        "description": "Look up a known fact: speed of light, pi, earth radius, "\n                       "days in a year.",\n        "parameters": {"query": "string - what to look up"},\n        "fn": _lookup,\n    },\n}\n\n\ndef build_tool_descriptions(tools):\n    """Render a tool registry as prompt text (Day 79)."""\n    lines = []\n    for name, spec in tools.items():\n        params = ", ".join(spec.get("parameters", {}))\n        lines.append("- " + name + "(" + params + "): " + spec["description"])\n    return "\\n".join(lines)\n\n\ndef safe_parse_json(text):\n    """Slice first \'{\' to last \'}\' and parse. Returns dict|None (Day 79)."""\n    start, end = text.find("{"), text.rfind("}")\n    if start == -1 or end == -1 or end < start:\n        return None\n    try:\n        data = json.loads(text[start:end + 1])\n    except (json.JSONDecodeError, ValueError):\n        return None\n    return data if isinstance(data, dict) else None\n\n# ── parsing the ReAct format ──────────────────────────────────────────────────\ndef _line_value(text, prefix):\n    """Text after the first line starting with prefix (case-insensitive), else \'\'."""\n    for line in text.splitlines():\n        if line.strip().lower().startswith(prefix.lower()):\n            return line.strip()[len(prefix):].strip()\n    return ""\n\n\ndef _after_marker(text, marker):\n    """Everything after marker (case-insensitive), or None if absent."""\n    idx = text.lower().find(marker.lower())\n    if idx == -1:\n        return None\n    return text[idx + len(marker):].strip()\n\n\ndef parse_react_step(text):\n    """Parse one ReAct step. NEVER raises.\n\n    Returns either:\n      {"type": "action", "thought": str, "tool": str, "input": dict}\n      {"type": "final",  "thought": str, "answer": str}\n    A reply with no recognisable Action falls back to a final answer holding\n    the raw text - so a malformed step still ends the loop cleanly.\n    """\n    thought = _line_value(text, "Thought:")\n    final = _after_marker(text, "Final Answer:")\n    if final is not None:\n        return {"type": "final", "thought": thought, "answer": final}\n    action = _line_value(text, "Action:")\n    if action:\n        args = safe_parse_json(_line_value(text, "Input:")) or {}\n        return {"type": "action", "thought": thought, "tool": action, "input": args}\n    return {"type": "final", "thought": thought, "answer": text.strip()}\n\n# ── formatting the trace (the scratchpad) ─────────────────────────────────────\ndef format_step(step):\n    """Render an action step back into ReAct text for the scratchpad."""\n    return ("Thought: " + step["thought"] + "\\n"\n            + "Action: " + step["tool"] + "\\n"\n            + "Input: " + json.dumps(step["input"]))\n\n\ndef format_observation(result):\n    """Render a tool result as an Observation line."""\n    return "Observation: " + str(result)\n\n\ndef build_react_prompt(task, tools, scratchpad):\n    """Build the [system, user] messages for one ReAct step."""\n    system = "\\n".join([\n        "You are a reasoning agent. Solve the task step by step using the "\n        "ReAct format: reason, act, observe, repeat.",\n        "",\n        "Available tools:",\n        build_tool_descriptions(tools),\n        "",\n        "On each turn reply in EXACTLY this format:",\n        "Thought: <your reasoning about what to do next>",\n        "Action: <one tool name from the list above>",\n        \'Input: {"<param>": "<value>"}\',\n        "",\n        "You will then receive an Observation with the tool\'s result.",\n        "When you can answer, reply instead with:",\n        "Thought: <your final reasoning>",\n        "Final Answer: <the answer>",\n    ])\n    user = "Task: " + str(task)\n    if scratchpad:\n        user = user + "\\n\\n" + scratchpad.rstrip()\n    user = user + "\\n\\nThought:"\n    return [{"role": "system", "content": system},\n            {"role": "user", "content": user}]\n\n# ── acting + calling the model ────────────────────────────────────────────────\ndef execute_action(step, tools):\n    """Run the tool named in a ReAct action step. Returns a string, never raises."""\n    name = step.get("tool")\n    if name not in tools:\n        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)\n    try:\n        return str(tools[name]["fn"](step.get("input", {})))\n    except Exception as exc:\n        return "Error running " + str(name) + ": " + str(exc)\n\n\ndef call_llm(messages, llm_fn=None):\n    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""\n    if llm_fn is not None:\n        return llm_fn(messages)\n    import ollama\n    resp = ollama.chat(model="llama3.2", messages=messages)\n    return resp["message"]["content"]\n\n# ── the ReAct loop ────────────────────────────────────────────────────────────\ndef run_react_agent(task, tools=None, llm_fn=None, max_iterations=10):\n    """Run the ReAct loop until a Final Answer or max_iterations.\n\n    Each turn: build a prompt from the task + scratchpad, ask the model for a\n    Thought/Action/Input, run the tool, append the step and its Observation to\n    the scratchpad, repeat. Feeding the growing scratchpad back is what lets the\n    model reason over its own earlier observations.\n\n    Returns {"answer", "thought", "trace", "iterations", "stopped"}.\n    """\n    if tools is None:\n        tools = DEFAULT_TOOLS\n    scratchpad = ""\n    trace = []\n    for i in range(max_iterations):\n        messages = build_react_prompt(task, tools, scratchpad)\n        step = parse_react_step(call_llm(messages, llm_fn=llm_fn))\n        if step["type"] == "final":\n            trace.append(step)\n            return {"answer": step["answer"], "thought": step["thought"],\n                    "trace": trace, "iterations": i + 1, "stopped": False}\n        result = execute_action(step, tools)\n        step["observation"] = result\n        trace.append(step)\n        scratchpad = scratchpad + format_step(step) + "\\n"\n        scratchpad = scratchpad + format_observation(result) + "\\n"\n    return {"answer": "Stopped: reached max_iterations without a final answer.",\n            "thought": "", "trace": trace,\n            "iterations": max_iterations, "stopped": True}\n\n# ── the ReAct agent as a class ────────────────────────────────────────────────\nclass ReactAgent:\n    """A reasoning agent using the ReAct loop.\n\n    Binds a tool registry and an optional llm_fn, runs tasks through\n    run_react_agent, and keeps a history of runs.\n\n    Example::\n\n        agent = ReactAgent(llm_fn=my_llm_fn)\n        result = agent.run("What is 12 * 12?")\n        print(result["answer"])\n        for step in result["trace"]:\n            print(step)\n    """\n\n    def __init__(self, tools=None, llm_fn=None, max_iterations=10):\n        # copy so add_tool never mutates the shared DEFAULT_TOOLS global (Day 79)\n        self.tools = dict(DEFAULT_TOOLS if tools is None else tools)\n        self._llm_fn = llm_fn\n        self.max_iterations = max_iterations\n        self._history = []\n\n    def add_tool(self, name, description, fn, parameters=None):\n        """Register a new tool; returns self."""\n        self.tools[name] = {"description": description,\n                            "parameters": parameters or {}, "fn": fn}\n        return self\n\n    def run(self, task):\n        """Run one task through the ReAct loop. Returns the result dict."""\n        result = run_react_agent(task, tools=self.tools, llm_fn=self._llm_fn,\n                                 max_iterations=self.max_iterations)\n        self._history.append({"task": task, "result": result})\n        return result\n\n    def history(self):\n        """Return a copy of the run history."""\n        return list(self._history)\n\n    def clear_history(self):\n        """Clear the run history in place."""\n        self._history.clear()\n'
from pathlib import Path
Path('react_agent.py').write_text(_SRC, encoding='utf-8')
print('react_agent.py written.')

In [ ]:

from react_agent import (
    DEFAULT_TOOLS, build_tool_descriptions, safe_parse_json,
    parse_react_step, format_step, format_observation, build_react_prompt,
    execute_action, call_llm, run_react_agent, ReactAgent,
)

def _make_mock_llm(script):
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn

# 1. parse_react_step
a = parse_react_step('Thought: add\nAction: calculator\nInput: {"expression": "2+2"}')
assert a['type'] == 'action' and a['tool'] == 'calculator' and a['input']['expression'] == '2+2'
f = parse_react_step('Thought: done\nFinal Answer: 4')
assert f['type'] == 'final' and f['answer'] == '4'
assert parse_react_step('rambling, no format')['type'] == 'final'
print("✅ parse_react_step (action / final / fallback)")

# 2. formatting
step = {'thought': 't', 'tool': 'calculator', 'input': {'expression': '2+2'}}
assert 'Action: calculator' in format_step(step) and '2+2' in format_step(step)
assert format_observation('4') == 'Observation: 4'
print("✅ format_step + format_observation")

# 3. build_react_prompt
msgs = build_react_prompt('t', DEFAULT_TOOLS, 'Observation: 4')
assert msgs[0]['role'] == 'system' and 'calculator' in msgs[0]['content']
assert '4' in msgs[1]['content']
print("✅ build_react_prompt (tools + scratchpad)")

# 4. execute_action
assert execute_action({'tool': 'calculator', 'input': {'expression': '6*7'}}, DEFAULT_TOOLS) == '42'
assert 'unknown tool' in execute_action({'tool': 'zzz', 'input': {}}, DEFAULT_TOOLS).lower()
assert 'error' in execute_action({'tool': 'calculator', 'input': {}}, DEFAULT_TOOLS).lower()
assert '3.14' in execute_action({'tool': 'lookup', 'input': {'query': 'pi'}}, DEFAULT_TOOLS)
print("✅ execute_action (tool / unknown / error / lookup)")

# 5. call_llm injection
assert call_llm([{'role': 'user', 'content': 'hi'}], llm_fn=lambda m: 'X') == 'X'
print("✅ call_llm (llm_fn injection)")

# 6. run_react_agent: act, observe, finish
script = ['Thought: I should add.\nAction: calculator\nInput: {"expression": "2+2"}',
          'Thought: done.\nFinal Answer: The answer is 4.']
out = run_react_agent('what is 2+2', DEFAULT_TOOLS, llm_fn=_make_mock_llm(script))
assert out['answer'] == 'The answer is 4.' and out['stopped'] is False
assert len(out['trace']) == 2 and out['trace'][0]['observation'] == '4'
print("✅ run_react_agent (act -> observe -> finish, trace recorded)")

# 7. observation fed back into the scratchpad
seen = []
def _spy(messages):
    seen.append(messages[1]['content'])
    if len(seen) == 1:
        return 'Thought: add.\nAction: calculator\nInput: {"expression": "2+2"}'
    return 'Thought: done.\nFinal Answer: 4'
run_react_agent('2+2', DEFAULT_TOOLS, llm_fn=_spy)
assert 'Observation: 4' in seen[1] and 'Observation' not in seen[0]
print("✅ run_react_agent (scratchpad feedback)")

# 8. max_iterations safeguard
never = _make_mock_llm(['Thought: loop.\nAction: calculator\nInput: {"expression": "1+1"}'])
loop = run_react_agent('x', DEFAULT_TOOLS, llm_fn=never, max_iterations=3)
assert loop['stopped'] is True and loop['iterations'] == 3
print("✅ run_react_agent (max_iterations stops runaway loop)")

# 9. ReactAgent
agent = ReactAgent(llm_fn=_make_mock_llm(script))
assert agent.run('2+2?')['answer'] == 'The answer is 4.'
agent.add_tool('shout', 'Uppercase.', lambda args: str(args['text']).upper(), {'text': 'string'})
assert 'shout' in agent.tools and 'shout' not in DEFAULT_TOOLS
assert len(agent.history()) == 1
agent.history().clear()
assert len(agent.history()) == 1
agent.clear_history()
assert len(agent.history()) == 0
print("✅ ReactAgent (run / add_tool / history / clear_history)")

print("\nReAct agent complete!")
